## Example 1: tenacity functionality test 1

In [ ]:
import tenacity
import time
import random

random.seed(20260730)

# Example 1: Basic retry functionality with decorator
@tenacity.retry(stop=tenacity.stop_after_attempt(3), 
                wait=tenacity.wait_fixed(0.1))
def unreliable_api_call(success_rate=0.6):
    """Simulate an unreliable API call that sometimes fails"""
    if random.random() > success_rate:
        raise ConnectionError("Network timeout")
    return {"status": "success", "data": {"user_count": 42}}

print("Basic retry functionality:")
try:
    result = unreliable_api_call(0.7)
    print(f"API call succeeded: {result}")
except Exception as e:
    print(f"API call failed after retries: {e}")
    raise

# Show retry configuration options
print(f"Available stop strategies: stop_after_attempt, stop_after_delay")
print(f"Available wait strategies: wait_fixed, wait_exponential, wait_random")
print(f"Retry decorator available: {hasattr(tenacity, 'retry')}")

## Example 2: tenacity functionality test 2

In [ ]:
import tenacity
import random

random.seed(20260730)

# Example 2: Advanced retry conditions and exception handling
@tenacity.retry(
    retry=tenacity.retry_if_exception_type((ConnectionError, TimeoutError)),
    stop=tenacity.stop_after_attempt(4),
    wait=tenacity.wait_exponential(multiplier=1, min=0.1, max=2)
)
def database_operation(operation_type="read"):
    """Simulate database operations with different failure modes"""
    failure_roll = random.random()
    
    if failure_roll < 0.3:
        raise ConnectionError("Database connection lost")
    elif failure_roll < 0.5:
        raise TimeoutError("Query timeout")
    elif failure_roll < 0.6:
        # This error should NOT trigger retry
        raise ValueError("Invalid query parameters")
    else:
        return f"Database {operation_type} completed successfully"

print("Conditional retry with exception types:")
outcomes = {}
for operation in ["read", "write", "update"]:
    try:
        result = database_operation(operation)
        outcomes[operation] = ("ok", database_operation.statistics["attempt_number"])
        print(f"✓ {result} (attempt {outcomes[operation][1]})")
    except ValueError as e:
        # Not in retry_if_exception_type: raised on the first attempt, never retried
        outcomes[operation] = ("not retried", database_operation.statistics["attempt_number"])
        print(f"✗ Non-retryable error: {e}")
    except tenacity.RetryError as e:
        # stop_after_attempt(4) ran out on ConnectionError/TimeoutError
        outcomes[operation] = ("gave up", e.last_attempt.attempt_number)
        print(f"✗ Gave up after {e.last_attempt.attempt_number} attempts: {e.last_attempt.exception()!r}")

assert outcomes == {"read": ("ok", 1), "write": ("ok", 1), "update": ("ok", 3)}, outcomes

print(f"\nRetry conditions: retry_if_exception_type, retry_if_result")
print(f"Wait strategies: exponential backoff available")

## Example 3: tenacity functionality test 3

In [ ]:
import tenacity
import random

random.seed(20260730)

# Example 3: Retry with callbacks and result conditions
def log_before(retry_state):
    """Callback before each attempt"""
    print(f"Starting attempt {retry_state.attempt_number}")

def log_after(retry_state):
    """Callback after each attempt"""
    if retry_state.outcome.failed:
        print(f"Attempt {retry_state.attempt_number} failed")
    else:
        print(f"Attempt {retry_state.attempt_number} succeeded")

@tenacity.retry(
    stop=tenacity.stop_after_attempt(5),
    wait=tenacity.wait_fixed(0.1),
    before=log_before,
    after=log_after,
    retry=tenacity.retry_if_result(lambda x: x is None)
)
def fetch_user_data(user_id):
    """Simulate fetching user data that sometimes returns None"""
    responses = [None, None, {"id": user_id, "name": "John", "active": True}]
    return random.choice(responses)

print("Retry with callbacks and result conditions:")
user_data = fetch_user_data(123)
print(f"\nFinal result: {user_data}")

print("\nExample 3 completed!")

## Example 4: tenacity functionality test 4

In [ ]:
import tenacity
import random

random.seed(20260730)

# Example 4: Custom wait strategies and wait combinations
# Custom wait strategy
class CustomWaitStrategy:
    """Custom wait strategy that increases wait time based on attempt number"""
    def __call__(self, retry_state):
        return retry_state.attempt_number * 0.1

@tenacity.retry(
    stop=tenacity.stop_after_attempt(3),
    wait=CustomWaitStrategy()
)
def service_with_custom_wait():
    """Service using custom wait strategy"""
    if random.random() < 0.7:
        raise Exception("Service temporarily unavailable")
    return "Service response"

# Combined wait strategies
@tenacity.retry(
    stop=tenacity.stop_after_attempt(3),
    wait=tenacity.wait_fixed(0.1) + tenacity.wait_random(0, 0.1),
    retry=tenacity.retry_if_exception_type(ConnectionError)
)
def api_with_combined_wait(endpoint):
    """API with combined wait strategies (fixed + random)"""
    if random.random() < 0.6:
        raise ConnectionError(f"Failed to connect to {endpoint}")
    return {"endpoint": endpoint, "status": "success"}

print("Custom wait strategies:")
try:
    result = service_with_custom_wait()
    print(f"Custom wait service: {result}")
except Exception as e:
    print(f"Custom wait failed: {e}")
    raise

print("\nCombined wait strategies:")
try:
    result = api_with_combined_wait("/api/users")
    print(f"API success: {result['status']}")
except Exception as e:
    print(f"API failed: {e}")
    raise

print("\nExample 4 completed!")

## Example 5: tenacity functionality test 5

In [ ]:
import random
random.seed(20260730)

import tenacity
import time

# Example 5: Production patterns and retry statistics
class ServiceClient:
    """Production service client with comprehensive retry logic"""
    
    def __init__(self):
        self.retry_stats = {"attempts": 0, "successes": 0, "failures": 0}
    
    @tenacity.retry(
        stop=tenacity.stop_after_attempt(5),
        wait=tenacity.wait_exponential(multiplier=0.5, min=0.1, max=5),
        retry=tenacity.retry_if_exception_type((ConnectionError, TimeoutError)),
        # a bound method's `self` arrives POSITIONALLY, so it is retry_state.args[0] (retry_state.kwargs has no 'self')
        before=lambda retry_state: setattr(retry_state.args[0], 'retry_stats', 
                                         {**retry_state.args[0].retry_stats, 'attempts': retry_state.args[0].retry_stats['attempts'] + 1}),
        reraise=True
    )
    def critical_operation(self, data):
        """Critical operation with production retry logic"""
        import random
        
        # Simulate various failure modes
        failure_type = random.random()
        if failure_type < 0.4:
            raise ConnectionError("Service unavailable")
        elif failure_type < 0.6:
            raise TimeoutError("Request timeout")
        elif failure_type < 0.7:
            # Permanent error - should not retry
            raise ValueError("Invalid data format")
        
        # Success case
        self.retry_stats["successes"] += 1
        return {"processed": len(data), "timestamp": time.time()}

# Batch processing with retry handling
client = ServiceClient()
test_data = [
    {"id": 1, "value": "data1"},
    {"id": 2, "value": "data2"},
    {"id": 3, "value": "data3"}
]

print("Production retry patterns:")
expected_outcomes = ["permanent", "success", "success"]
results = []
for i, data in enumerate(test_data):
    try:
        result = client.critical_operation([data])
    except ValueError as e:
        assert expected_outcomes[i] == "permanent" and str(e) == "Invalid data format", f"Batch {i+1}: {e!r}"
        results.append({"success": False, "error": "permanent", "message": str(e)})
        print(f"✓ Batch {i+1} not retried, as expected: simulated permanent {type(e).__name__} {str(e)!r}")
    except (ConnectionError, TimeoutError) as e:
        # reraise=True: once stop_after_attempt(5) is reached, the last retryable error surfaces as itself
        assert expected_outcomes[i] == "retry_exhausted", f"Batch {i+1}: retries exhausted, expected {expected_outcomes[i]}: {e!r}"
        results.append({"success": False, "error": "retry_exhausted", "message": str(e)})
        print(f"✓ Batch {i+1} gave up after 5 attempts, as expected: {type(e).__name__} {str(e)!r}")
    else:
        assert expected_outcomes[i] == "success", f"Batch {i+1} succeeded, expected {expected_outcomes[i]}"
        results.append({"success": True, "result": result})
        print(f"✓ Batch {i+1} processed successfully")

print(f"\nRetry Statistics:")
print(f"Total attempts: {client.retry_stats['attempts']}")
print(f"Successes: {client.retry_stats['successes']}")
print(f"Success rate: {len([r for r in results if r['success']])}/{len(results)}")
print(f"Production features: exponential backoff, exception filtering, statistics")
# every call ran the `before` hook at least once, and no batch can take more than 5 attempts
assert len(test_data) <= client.retry_stats['attempts'] <= 5 * len(test_data), client.retry_stats
assert client.retry_stats['successes'] == len([r for r in results if r['success']])
# and the seeded run exactly (measured, see expected_outcomes): 5 attempts, 2 successes
assert client.retry_stats == {"attempts": 5, "successes": 2, "failures": 0}, client.retry_stats